# Automated Solid Waste Classification: Exploratory Data Analysis & Baseline Benchmarking
**Project ID**: DL-01  
**Domain**: Part B: Deep Learning (Computer Vision & Sustainability)  
**Target Classes (6)**: Cardboard, Glass, Metal, Paper, Plastic, Trash  

---

## 1. Problem Formulation & Environmental Significance
Municipal solid waste management is an urgent environmental challenge. Global municipal solid waste generation is projected to reach 3.4 billion tonnes by 2050. Automated optical sorting systems deployed in Material Recovery Facilities (MRFs) require robust visual recognition models to divert recyclable materials from landfills, reduce cross-contamination in recycling bales, and protect human sorters from biological and mechanical hazards.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torchvision.transforms as transforms

# Ensure project modules are on path
sys.path.append(os.path.abspath('..'))
from src.data_loader import CLASS_NAMES, NUM_CLASSES, generate_synthetic_waste_dataset
from src.model import WasteClassifierCNN
from src.preprocessor import get_transforms

print(f"Target classes ({NUM_CLASSES}):", CLASS_NAMES)

## 2. Dataset Ingestion & Class Distribution Analysis
We inspect the sample distribution across the six material streams.

In [2]:
# Benchmark class frequencies in Garbage Classification dataset
distribution_data = {
    'Class': ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash'],
    'Benchmark_Count': [403, 501, 410, 594, 482, 137],
    'Recyclability': ['Recyclable (Blue)', 'Recyclable (Green)', 'Recyclable (Yellow)', 'Recyclable (Blue)', 'Recyclable (Yellow)', 'Landfill (Grey)']
}
df_dist = pd.DataFrame(distribution_data)
df_dist['Percentage'] = (df_dist['Benchmark_Count'] / df_dist['Benchmark_Count'].sum()) * 100

plt.figure(figsize=(9, 4.5))
sns.barplot(data=df_dist, x='Class', y='Benchmark_Count', palette='viridis')
plt.title('Sample Distribution Across Solid Waste Categories', fontsize=12, pad=10)
plt.xlabel('Material Stream')
plt.ylabel('Image Count')
plt.grid(axis='y', alpha=0.3)
for idx, row in df_dist.iterrows():
    plt.text(idx, row['Benchmark_Count'] + 10, f"{row['Percentage']:.1f}%", ha='center', fontsize=9)
plt.tight_layout()
plt.show()

## 3. Data Augmentation & Preprocessing Pipeline Visualization
Visualizing the stochastic transformations (random horizontal flips, color jitter, affine rotations) designed to build invariant feature representations under variable MRF illumination conditions.

In [3]:
samples = generate_synthetic_waste_dataset(n_samples_per_class=1, image_size=(128, 128), seed=101)
aug_transform = get_transforms(image_size=(128, 128), is_training=True)

fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes = axes.flatten()

for i, (img, label) in enumerate(samples[:6]):
    # Inverse transform to plot
    t_img = aug_transform(img)
    np_img = t_img.permute(1, 2, 0).numpy()
    # Denormalize
    np_img = np_img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
    np_img = np.clip(np_img, 0, 1)
    axes[i].imshow(np_img)
    axes[i].set_title(f"{CLASS_NAMES[label]}")
    axes[i].axis('off')

plt.suptitle('Sample Augmented Waste Images', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Deep Learning Model Architecture Inspection
Inspecting the parameter count and tensor shapes through the custom 4-stage convolutional hierarchy.

In [4]:
model = WasteClassifierCNN(num_classes=6, backbone='custom')
total_p, train_p = model.count_parameters()
print(f"Total Model Parameters: {total_p:,}")
print(f"Trainable Parameters:  {train_p:,}")

# Forward pass verification with dummy batch
dummy_batch = torch.randn(8, 3, 224, 224)
with torch.no_grad():
    out = model(dummy_batch)
print("Output Logits Shape:", out.shape)
print("Sample Softmax Probabilities:\n", torch.softmax(out[:2], dim=1).numpy().round(3))

## 5. Conclusions & Next Steps
- The 4-stage CNN architecture provides sufficient capacity while remaining compact enough for edge deployment.
- Preprocessing transforms effectively mitigate illumination variations and material orientation noise.
- Next step: Full training and evaluation with confusion matrix logging and Streamlit UI deployment.